# California Public Expenditure Analysis
**Exploratory Data Analysis | 2013-2023**

This notebook analyzes city-level public expenditure data across California's four regions:
Northern California, Southern California, Central California, and Sierra Nevada.

**Data source:** California State Controller's Office  
**Files used:** `City_Expenditures_Per_Capita.csv`, `city_to_region.csv`

## 1. Load & Merge Data

In [ ]:
import pandas as pd

expenditure = pd.read_csv('City_Expenditures_Per_Capita.csv')
regions = pd.read_csv('city_to_region.csv')

df = expenditure.merge(regions, on='Entity Name', how='inner')
df = df[(df['Fiscal Year'] >= 2013) & (df['Fiscal Year'] <= 2023)]
df = df.dropna(subset=['Expenditures Per Capita'])

print(f'Records: {len(df):,}')
print(f'Cities:  {df["Entity Name"].nunique()}')
print(f'Years:   {df["Fiscal Year"].min()} - {df["Fiscal Year"].max()}')

table_style = [
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left'), ('padding', '6px 12px'), ('background', '#f0f4f8'), ('color', '#212529')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('color', '#212529'), ('background', '#ffffff')]},
    {'selector': '', 'props': [('border-collapse', 'collapse')]}
]

df.head().style.set_table_styles(table_style)

## 2. Regional Overview
Total expenditure and 2023 population by region.

In [ ]:
totals = df.groupby('Region')['Total Expenditures'].sum().sort_values(ascending=False)
pop_2023 = df[df['Fiscal Year'] == 2023].groupby('Region')['Estimated Population'].sum()

overview = pd.DataFrame({
    'Total Expenditure ($B)': (totals / 1e9).round(1).apply(lambda x: f'{x:.1f}'),
    'Share (%)': (totals / totals.sum() * 100).round(1).apply(lambda x: f'{x:.1f}'),
    'Population 2023 (M)': (pop_2023 / 1e6).round(2).apply(lambda x: f'{x:.2f}')
}).reset_index()

overview.style.set_table_styles([
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left'), ('padding', '6px 12px'), ('background', '#f0f4f8'), ('color', '#212529')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('color', '#212529'), ('background', '#ffffff')]},
    {'selector': '', 'props': [('border-collapse', 'collapse')]}
]).hide(axis='index')

## 3. Mean vs Median Per Capita
When mean is much higher than median, outliers are skewing the average.

In [ ]:
stats = df.groupby('Region')['Expenditures Per Capita'].agg(
    Mean='mean',
    Median='median',
    Std='std'
).round(0).astype(int)

stats['Mean/Median Ratio'] = (stats['Mean'] / stats['Median']).round(1)
stats['Outlier Warning'] = stats['Mean/Median Ratio'].apply(
    lambda x: 'Yes - investigate' if x > 3 else 'No'
)
stats.reset_index().style.set_table_styles([
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left'), ('padding', '6px 12px'), ('background', '#f0f4f8'), ('color', '#212529')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('color', '#212529'), ('background', '#ffffff')]},
    {'selector': '', 'props': [('border-collapse', 'collapse')]}
]).hide(axis='index')

## 4. Identify Outliers
Top 10 highest per-capita cities across all years. These can distort regional averages.

In [ ]:
top10 = (
    df[['Entity Name', 'Fiscal Year', 'Expenditures Per Capita', 'Estimated Population', 'Region']]
    .sort_values('Expenditures Per Capita', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10['Expenditures Per Capita'] = top10['Expenditures Per Capita'].apply(lambda x: f'${x:,.0f}')
top10['Estimated Population'] = top10['Estimated Population'].apply(lambda x: f'{x:,.0f}')
top10.style.set_table_styles([
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left'), ('padding', '6px 12px'), ('background', '#f0f4f8'), ('color', '#212529')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('color', '#212529'), ('background', '#ffffff')]},
    {'selector': '', 'props': [('border-collapse', 'collapse')]}
])

## 5. Per Capita Growth by Region (2013 to 2023)
Using median to avoid outlier distortion.

In [ ]:
y2013 = df[df['Fiscal Year'] == 2013].groupby('Region')['Expenditures Per Capita'].median()
y2023 = df[df['Fiscal Year'] == 2023].groupby('Region')['Expenditures Per Capita'].median()

growth = pd.DataFrame({'2013 Median': y2013, '2023 Median': y2023})
growth['Change (%)'] = ((growth['2023 Median'] / growth['2013 Median'] - 1) * 100).round(0).astype(int)
growth['2013 Median'] = growth['2013 Median'].apply(lambda x: f'${x:,.0f}')
growth['2023 Median'] = growth['2023 Median'].apply(lambda x: f'${x:,.0f}')
growth['Change (%)'] = growth['Change (%)'].apply(lambda x: f'+{x}%' if x > 0 else f'{x}%')
growth.reset_index().style.set_table_styles([
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left'), ('padding', '6px 12px'), ('background', '#f0f4f8'), ('color', '#212529')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('color', '#212529'), ('background', '#ffffff')]},
    {'selector': '', 'props': [('border-collapse', 'collapse')]}
]).hide(axis='index')

## 6. Top 5 & Bottom 5 Cities per Region
Excluding Vernon and Industry (industrial enclaves with ~100 residents) as they are not representative.

In [ ]:
from IPython.display import display, HTML

OUTLIERS = ['Vernon', 'Industry']
filtered = df[~df['Entity Name'].isin(OUTLIERS)]

city_avg = (
    filtered.groupby(['Entity Name', 'Region'])['Expenditures Per Capita']
    .mean()
    .reset_index()
    .rename(columns={'Expenditures Per Capita': 'Avg Per Capita'})
)

table_style = [
    {'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left'), ('padding', '6px 12px'), ('background', '#f0f4f8'), ('color', '#212529')]},
    {'selector': 'td', 'props': [('padding', '6px 12px'), ('color', '#212529'), ('background', '#ffffff')]},
    {'selector': '', 'props': [('border-collapse', 'collapse')]}
]

for region in sorted(city_avg['Region'].unique()):
    rc = city_avg[city_avg['Region'] == region].sort_values('Avg Per Capita', ascending=False)
    top5 = rc.head(5)[['Entity Name', 'Avg Per Capita']].copy().reset_index(drop=True)
    bot5 = rc.tail(5)[['Entity Name', 'Avg Per Capita']].copy().reset_index(drop=True)
    top5['Avg Per Capita'] = top5['Avg Per Capita'].apply(lambda x: f'${x:,.0f}')
    bot5['Avg Per Capita'] = bot5['Avg Per Capita'].apply(lambda x: f'${x:,.0f}')

    top5_html = top5.style.set_table_styles(table_style).hide(axis='index').to_html()
    bot5_html = bot5.style.set_table_styles(table_style).hide(axis='index').to_html()

    display(HTML(f'''
        <h4 style="color:#1a1a2e;margin-top:28px;margin-bottom:12px">{region}</h4>
        <div style="display:flex;gap:40px;align-items:flex-start">
            <div>
                <p style="color:#555;margin:0 0 6px;font-size:13px">Top 5</p>
                {top5_html}
            </div>
            <div>
                <p style="color:#555;margin:0 0 6px;font-size:13px">Bottom 5</p>
                {bot5_html}
            </div>
        </div>
    '''))

## 7. Key Takeaways

1. **Southern California's average is misleading.** Its mean per-capita (11,762) is nearly 9x its median (1,343) due to Vernon, an industrial city of ~100 residents spending over 2M per capita. Median is the right metric here.

2. **All regions grew significantly.** Median per-capita spending rose 39-69% across all regions from 2013 to 2023, with a visible uptick in 2020-2021 consistent with pandemic-era emergency spending.

3. **City size predicts per-capita spend more than region does.** The top spenders in every region are small municipalities, resort towns (Mammoth Lakes), tech hubs (Mountain View), or coastal cities (Carmel-By-The-Sea), with high fixed costs relative to their populations.

4. **Within-region variation is extreme.** In Northern California, Mountain View (45,645) spends 69x more per capita than Citrus Heights (660). Regional averages alone are not sufficient for meaningful comparison.